# CS 3892 / 5892 — Session 10 · Project lightning talks, and nuXmv in depth

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-09-29-project-lightning-talks.ipynb)

**Tuesday, September 29, 2026.** Today is the project proposal talks. These are the
**backup slides** — if the talks finish early we go deeper into the model checker
that HW2 runs on, using the same counter as session 9.

Everything here runs **NuSMV** (free, LGPL), fetched from FBK by the setup cells.
One example (`02_add2_integer.smv`) needs **nuXmv**, which is licensed separately;
its recorded output is shown instead of run.

## Setup

Run these two once.

In [ ]:
# --- Setup: find the repo (clone on Colab), install Z3, define helpers -------
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-09-29-project-lightning-talks"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

try:
    import z3
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "z3-solver"], check=True)
    import z3
print("Z3", z3.get_version_string())

SM = ROOT / "sessions" / SESSION / "smt2"
PYD = ROOT / "sessions" / SESSION / "python"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    print(f"--- {pathlib.Path(path).name} " + "-" * max(0, 60 - len(pathlib.Path(path).name)))
    print(pathlib.Path(path).read_text().rstrip())
    print()

def run(path, show_source=True):
    """Run one example and stream its output. Raises if it does not pass.

    .smt2 goes through scripts/run_smt2.py, which checks the file's own
    `; EXPECT:` contract. .py is executed directly and asserts internally.
    The pip wheel for Z3 ships no `z3` CLI, which is why .smt2 is run through
    the Python bindings rather than a shell command -- identical everywhere.
    """
    path = pathlib.Path(path)
    if show_source:
        show(path)
    cmd = ([sys.executable, "scripts/run_smt2.py", str(path)] if path.suffix == ".smt2"
           else [sys.executable, str(path)])
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout.rstrip())
    if r.stderr.strip():
        print(r.stderr.rstrip(), file=sys.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"{path} failed")
    return r.stdout

print("ready — helpers: show(path), run(path)")

In [ ]:
# --- NuSMV: fetch the official FBK build if it is not already installed ------
# NuSMV is free (LGPL) from nusmv.fbk.eu. nuXmv is license-gated, so it is not
# installed here -- the smvis link above covers the browser route.
# (Same cell as session 9.) 2.7.1 is tried first; it needs glibc >= 2.38 and libedit, which an older Colab
# image may lack, so the static 2.6.0 build is the fallback. Same verdicts.
import re, shutil, subprocess, os, pathlib

SMV = ROOT / "sessions" / SESSION / "smv"
BUILDS = [("2.7.1", "https://nusmv.fbk.eu/distrib/2.7.1/NuSMV-2.7.1-linux64.tar.xz", "J"),
          ("2.6.0", "https://nusmv.fbk.eu/distrib/NuSMV-2.6.0-linux64.tar.gz",       "z")]

def _works(exe):
    """True only if this binary actually model-checks the file -- a missing
    library makes it exit 127 with an error that still contains "NuSMV"."""
    try:
        r = subprocess.run([exe, str(SMV / "01_three_engines.smv")], capture_output=True, text=True, timeout=60)
        return "-- specification" in r.stdout or "-- invariant" in r.stdout
    except Exception:
        return False

NUSMV = shutil.which("NuSMV")
if not (NUSMV and _works(NUSMV)):
    NUSMV = None
    if os.geteuid() == 0 and shutil.which("apt-get"):      # Colab runs as root
        subprocess.run("apt-get install -y -qq libedit2 >/dev/null 2>&1", shell=True)
    for ver, url, z in BUILDS:
        dest = pathlib.Path.home() / ".local" / f"nusmv-{ver}"
        dest.mkdir(parents=True, exist_ok=True)
        subprocess.run(f"curl -sSL {url} | tar -x{z} -C {dest} --strip-components=1",
                       shell=True, check=True)
        if _works(str(dest / "bin" / "NuSMV")):
            NUSMV = str(dest / "bin" / "NuSMV")
            break
        print(f"NuSMV {ver} will not start on this machine -- trying the next build")
assert NUSMV, "could not install NuSMV"
print("NuSMV:", NUSMV)

def smv(path, commands=None):
    """Run NuSMV on a file -- batch mode, or an interactive command script."""
    if commands is None:
        r = subprocess.run([NUSMV, str(path)], capture_output=True, text=True)
    else:
        r = subprocess.run([NUSMV, "-int", str(path)], input="\n".join(commands + ["quit"]) + "\n",
                           capture_output=True, text=True)
    out = "\n".join(l for l in (r.stdout + r.stderr).splitlines() if not l.startswith("***"))
    return out
print("ready -- smv(path) or smv(path, [commands])")


## 1. One property, three engines

Backup slide 2. `INVARSPEC P` says *P holds in every reachable state* — the
invariant of session 9. Three ways to check it:

| engine | NuSMV command | what it can conclude |
|---|---|---|
| BDDs | `go` ; `check_invar` | computes the reachable set exactly, then compares |
| BMC / k-induction | `go_bmc` ; `check_invar_bmc -a een-sorensson -k K` | a bug within K steps — or a proof, if the property is K-inductive |
| IC3 (nuXmv only) | `go_msat` ; `check_invar_ic3` | builds an inductive invariant for you |

In [ ]:
F = SMV / "01_three_engines.smv"
show(F)
bdd = smv(F, ["go", 'check_invar -p "x <= 10"', 'check_invar -p "x <= 9"'])
print(bdd[:600])
assert re.search(r"invariant x <= 10\s+is true", bdd) and re.search(r"invariant x <= 9\s+is false", bdd)

for k in (9, 10):
    out = smv(F, ["go_bmc", f'check_invar_bmc -a een-sorensson -k {k} -p "x <= 9"'])
    last = [l for l in out.splitlines() if l.strip().startswith("--") or "-- " in l][-1]
    print(f"k = {k}:", last.split("> ")[-1])
    if k == 9:
        assert "cannot prove" in out, out
    else:
        assert re.search(r"invariant x <= 9\s+is false", out), out

ind = smv(F, ["go_bmc", 'check_invar_bmc -a een-sorensson -k 12 -p "x <= 10"'])
assert re.search(r"invariant x <= 10\s+is true", ind), ind
print("x <= 10 proved by k-induction at bound 0 -- it was already inductive (session 9, slide 18)")

## 2. The add-2 counter over the integers — nuXmv

Backup slide 3. Session 9's add-2 counter, now with `x : integer` — an infinite
state space. NuSMV cannot represent it (the cell below shows it refusing); nuXmv's
IC3 engine proves `x != 5` **without being told the strengthening**. Recorded output
from nuXmv 2.2.0:

```
nuXmv > go_msat
nuXmv > check_invar_ic3
-- invariant x != 5  is true
```

In [ ]:
F = SMV / "02_add2_integer.smv"
show(F)
out = smv(F)
print(out[:300])
assert "integer" in out and "undefined" in out, out
print("NuSMV rejects `integer` as expected -- this file is for nuXmv")

## 3. A first look at LTL — Thursday's topic

Backup slide 5. Same counter, three LTL properties. The false one comes back with a
**lasso**: a prefix and a loop that repeats forever (`-- Loop starts here`).

In [ ]:
F = SMV / "03_counter_ltl.smv"
show(F)
out = smv(F)
print("\n".join(l for l in out.splitlines() if l.startswith("-- specification") or "Loop starts" in l))
verdicts = re.findall(r"-- specification\s+(.*?)\s+is (true|false)", out)
assert [v for _, v in verdicts] == ["true", "true", "false"], verdicts
assert "Loop starts here" in out
print("verdicts as on the slide, and the counterexample is a lasso")